# 🏨 AGODA Price Crawler

Chạy lần lượt các cell từ trên xuống: **① Cấu hình → ② Đọc input → ③ Crawl → ④ Xem kết quả**.

**Đổi nguồn input:** sửa `INPUT_MODE` ở cell ① — `"gsheet"` (Google Sheet online) hoặc `"offline"` (file CSV/XLSX trên máy).

**Format file offline:** chỉ cần **3 cột đầu theo đúng thứ tự** `hotel_name, hotel_url, room_type` (tên cột không quan trọng, chỉ cần đúng thứ tự). Có file mẫu ở `input/TEMPLATE_hotels.csv`.

**Output** nằm trong `results/agoda/<RUN_NAME>/` — mỗi notebook 1 thư mục riêng (đặt `RUN_NAME` ở cell ①, mỗi notebook 1 tên khác nhau để không ghi đè lẫn nhau):
- `FINAL_<YYYYMMDD>.csv` — kết quả cuối
- `TEMP_agoda.csv` — checkpoint: lỡ tắt giữa chừng, chạy lại cell ③ sẽ tự resume phần chưa xong — hotel **chưa từng cào** sẽ chạy trước, hotel còn NA/SOLD OUT retry sau

Cache warm (`results/agoda/captures/`) vẫn dùng chung giữa các notebook nên không tốn thêm thời gian warm.

In [1]:
# ════════════════ ① CẤU HÌNH ════════════════

# ── Tên run: kết quả nằm RIÊNG trong results/agoda/<RUN_NAME>/ ──
# ⚠️ Mỗi notebook phải đặt 1 tên KHÁC nhau (notebook này "run2", notebook kia "run1")
RUN_NAME = "run2"

# ── Nguồn input: "gsheet" (online) hoặc "offline" (file trên máy) ──
INPUT_MODE = "gsheet"

# Dùng khi INPUT_MODE = "gsheet" (gid của tab được tự lấy từ URL)
GSHEET_URL = "https://docs.google.com/spreadsheets/d/1SnpIayEwkMMaLow2ImZLS9enmn7A80LB0IeQMAkemio/edit?gid=1827224235#gid=1827224235"

# Dùng khi INPUT_MODE = "offline" — đường dẫn tuyệt đối, hoặc tương đối so với 31.crawl-tool
# ⚠️ File phải có 3 cột đầu là (tên KS, URL, loại phòng) — file "TEMP_*" là checkpoint OUTPUT, không phải input!
OFFLINE_FILE = "input/v"

# ── Tham số crawl ──
WEEKS      = 6      # số tuần cần crawl
MAX_HOTELS = 0      # 0 = crawl tất cả; đặt 5 để test nhanh 5 khách sạn đầu
SHARD      = ""     # "" = không chia; "1/3" = chạy phần 1 trong 3 phần (chạy lần lượt 1/3, 2/3, 3/3)

In [2]:
# ════════════════ ② ĐỌC INPUT ════════════════
import os, sys

if "ROOT" not in globals():                    # giữ nguyên ROOT khi chạy lại cell
    ROOT = os.path.abspath("")                 # .../31.crawl-tool (nơi đặt notebook này)
assert os.path.isdir(os.path.join(ROOT, "crawler")), (
    f"Không tìm thấy package `crawler` trong {ROOT} — hãy mở notebook từ thư mục 31.crawl-tool")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import crawler
from crawler.hotels_io import read_hotels

if INPUT_MODE == "gsheet":
    INPUT = GSHEET_URL
    print("📡 Input: Google Sheet online")
else:
    INPUT = OFFLINE_FILE if os.path.isabs(OFFLINE_FILE) else os.path.join(ROOT, OFFLINE_FILE)
    assert os.path.exists(INPUT), f"Không tìm thấy file: {INPUT}"
    print(f"📁 Input: file offline — {INPUT}")

hotels = read_hotels(INPUT)
print(f"✅ Đọc được {len(hotels)} khách sạn. 5 dòng đầu:")
for name, url, room in hotels[:5]:
    print(f"   • {name} — {room}")

📡 Input: Google Sheet online
✅ Đọc được 98 khách sạn. 5 dòng đầu:
   • A25 — Superior 2 giường đơn (Superior Twins)
   • Alagon City Hotel & Spa — Superior Twin Internal Window
   • Altara Suites — Peace Suites 1 Phòng Ngủ (Peace Suites One Bedroom)
   • Bel Marina Hoi An Resort — Phòng Loại Sang Thủ Tướng Hướng Vườn (Premier Deluxe Garden View Room)
   • Bến Thành Boutique Hotel — Phòng Boutique (Boutique Room)


In [3]:
# (TÙY CHỌN) Tải Google Sheet về file offline — lần sau chỉ cần đổi INPUT_MODE = "offline"
import pandas as pd
from crawler.hotels_io import _gsheet_url

os.makedirs(os.path.join(ROOT, "input"), exist_ok=True)
dest = os.path.join(ROOT, "input", "agoda_hotels.csv")
pd.read_csv(_gsheet_url(GSHEET_URL)).to_csv(dest, index=False, encoding="utf-8-sig")
print(f"💾 Đã lưu bản offline: {dest}")

💾 Đã lưu bản offline: /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/input/agoda_hotels.csv


In [4]:
# ════════════════ ③ CRAWL ════════════════
OUTDIR = os.path.join(ROOT, "results", "agoda", RUN_NAME)   # mỗi notebook 1 thư mục kết quả riêng
os.makedirs(OUTDIR, exist_ok=True)
os.chdir(OUTDIR)                     # output (FINAL_*.csv, TEMP_agoda.csv) nằm ở đây

kwargs = dict(
    site="agoda",                    # direct replay (nhanh) + Camoufox warm
    input=INPUT,
    weeks=WEEKS,
    # cache warm dùng CHUNG cho mọi notebook (đỡ warm lại) — chỉ kết quả là tách riêng
    capture_dir=os.path.join(ROOT, "results", "agoda", "captures"),
)
if MAX_HOTELS:
    kwargs["max"] = MAX_HOTELS
if SHARD:
    kwargs["shard"] = SHARD

await crawler.arun(**kwargs)         # notebook cho phép await trực tiếp

📂 Resume: 98 rows from TEMP_agoda.csv
🚀 AGODA crawl | 98 hotels × 6w | direct+fallback | engine=camoufox | W1=2026-07-25
✔️  1/98 A25 — complete, skip
✔️  2/98 Alagon City Hotel & Spa — complete, skip
✔️  3/98 Altara Suites — complete, skip
✔️  4/98 Bel Marina Hoi An Resort — complete, skip
✔️  5/98 Bến Thành Boutique Hotel — complete, skip
✔️  6/98 Central - Deluxe Có ban công Hướng phố (Deluxe Balcony City View) - 25 — complete, skip
✔️  7/98 Central - Phòng Deluxe 2 giường có cửa sổ (Deluxe Twin Room With Window) - 25 — complete, skip
✔️  8/98 Central - Phòng Deluxe Có Giường Cỡ King Và Cửa Sổ (Deluxe King Room with Window) - 25 — complete, skip
✔️  9/98 Central - Phòng Gia Đình - 2 Giường (Family Room -- 2 Bed) - 50 — complete, skip
✔️  10/98 Central - Phòng Premier Có Bồn Tắm (Premier with Bathtub) - 40 — complete, skip
✔️  11/98 Central - Phòng Premier Hướng phố (Premier City View Room) - 30 — complete, skip
✔️  13/98 Central - Phòng Superior (Superior Room) - 15 — complete, skip

'FINAL_20260720.csv'

In [5]:
# ════════════════ ④ XEM KẾT QUẢ ════════════════
import glob
import pandas as pd

OUTDIR = os.path.join(ROOT, "results", "agoda", RUN_NAME)
files = sorted(glob.glob(os.path.join(OUTDIR, "FINAL_*.csv")))
assert files, "Chưa có file FINAL nào — hãy chạy cell ③ trước."
latest = files[-1]
df = pd.read_csv(latest)
print(f"📄 {latest} — {len(df)} dòng")
df.head(20)

📄 /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/results/agoda/run2/FINAL_20260720.csv — 98 dòng


,hotel_name,room_type,price_w1,price_w2,price_w3,price_w4,price_w5,price_w6
0,A25,Superior 2 giường đơn (Superior Twins),"945,767","945,767","1,024,581","1,024,581","945,767","1,024,581"
1,Alagon City Hotel & Spa,Superior Twin Internal Window,"1,659,240","1,659,240","1,804,908","1,813,095","1,659,240","1,284,458"
2,Altara Suites,Peace Suites 1 Phòng Ngủ (Peace Suites One Bed...,"2,259,524","2,607,143","2,607,143","2,607,143","2,428,571","2,428,571"
3,Bel Marina Hoi An Resort,Phòng Loại Sang Thủ Tướng Hướng Vườn (Premier ...,"2,448,149","2,448,149","3,076,190","3,346,032","2,448,149","2,448,149"
4,Bến Thành Boutique Hotel,Phòng Boutique (Boutique Room),"1,289,683","1,289,683","1,432,981","1,472,222","1,388,889","1,388,889"
5,Central - Deluxe Có ban công Hướng phố (Deluxe...,Deluxe Có ban công Hướng phố (Deluxe Balcony C...,"1,740,212","1,845,679","1,866,314","1,866,314","1,866,314","1,866,314"
6,Central - Phòng Deluxe 2 giường có cửa sổ (Del...,Phòng Deluxe 2 giường có cửa sổ (Deluxe Twin R...,"1,582,011","1,265,608","1,311,464","1,311,464","1,311,464","1,311,464"
7,Central - Phòng Deluxe Có Giường Cỡ King Và Cử...,Phòng Deluxe Có Giường Cỡ King Và Cửa Sổ (Delu...,"1,265,608","1,687,478","1,311,464","1,311,464","1,311,464","1,311,464"
8,Central - Phòng Gia Đình - 2 Giường (Family Ro...,Phòng Gia Đình - 2 Giường (Family Room -- 2 Bed),"2,947,186","2,947,186","2,874,333","2,874,333","2,774,250","2,774,250"
9,Central - Phòng Premier Có Bồn Tắm (Premier wi...,Phòng Premier Có Bồn Tắm (Premier with Bathtub),"2,373,016","2,373,016","2,471,605","2,471,605","2,471,605","2,471,605"
